# Panel G no-ULC / no-Kinnex sensitivity run (HPC)

Goal: regenerate the Panel G transcript concordance counts after excluding CAT transcript models whose gene/transcript biotype is `unknown_likely_coding`.

The filtering and re-comparison now run in the Nextflow workflow via `--filter_cat_gff true`. This notebook is intentionally lightweight: it prints the recommended workflow command, checks that the filtered workflow outputs exist, and renders the all-model vs no-ULC Panel G comparison.

Expected copy-back folder after completion: the filtered workflow `intermediate_spreadsheets/intron_chain/` plus this notebook's `intermediate_spreadsheets/kinnex_sensitivity/` render outputs.


In [ ]:
from pathlib import Path
import os
import textwrap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_DIR = Path(os.getenv('HPRC_QC_REPO_DIR', '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc'))
BASE_OUTPUT_DIR = Path(os.getenv('HPRC_QC_BASE_OUTPUT_DIR', '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results'))
NO_ULC_OUTPUT_DIR = Path(os.getenv('HPRC_QC_NO_ULC_OUTPUT_DIR', '/hps/nobackup/flicek/ensembl/genebuild/jackt/hprc/hprc-qc/results_no_unknown_likely_coding'))

BASE_INTRON_DIR = BASE_OUTPUT_DIR / 'intermediate_spreadsheets' / 'intron_chain'
NO_ULC_INTRON_DIR = NO_ULC_OUTPUT_DIR / 'intermediate_spreadsheets' / 'intron_chain'
WORK_DIR = BASE_OUTPUT_DIR / 'intermediate_spreadsheets' / 'kinnex_sensitivity'
FIG_DIR = WORK_DIR / 'figures'
WORK_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_DIR:', REPO_DIR)
print('BASE_OUTPUT_DIR:', BASE_OUTPUT_DIR)
print('NO_ULC_OUTPUT_DIR:', NO_ULC_OUTPUT_DIR)
print('BASE_INTRON_DIR:', BASE_INTRON_DIR, BASE_INTRON_DIR.exists())
print('NO_ULC_INTRON_DIR:', NO_ULC_INTRON_DIR, NO_ULC_INTRON_DIR.exists())
print('WORK_DIR:', WORK_DIR)


In [ ]:
# Recommended Nextflow command for the filtered Panel G run.
# Fill in --input/--ensg_lookup and profile/container options as usual for the cluster.
cmd = f'''
cd {REPO_DIR / 'nextflow/pipelines/ensembl_cat_comparison'}
nextflow run main.nf \
  --input <assemblies.csv> \
  --ensg_lookup <transcript_to_ensg_lookup.tsv> \
  --outdir {NO_ULC_OUTPUT_DIR} \
  --comparison_script {REPO_DIR / 'nextflow/pipelines/ensembl_cat_comparison/bin/hprc_ensembl_cat_overlap.py'} \
  --filter_cat_gff true \
  --filter_cat_exclude_biotypes unknown_likely_coding \
  --run_transcript_concordance true \
  --run_gene_transcript_counts true \
  --run_cat_gene_transcript_counts true \
  --run_coding_integrity false \
  --run_gene_presence false \
  --run_multi_mapping false \
  --run_grch38_divergence false \
  --run_gff_feature_metrics false \
  --run_gffcompare false \
  --run_aggregate true \
  --aggregate_gene_presence false \
  --aggregate_sankey false \
  --aggregate_coding_integrity false \
  --aggregate_transcript_counts false \
  --aggregate_divergence false \
  --aggregate_concordance_vs_ref false \
  --aggregate_intron_chain_by_biotype true \
  -profile slurm
'''.strip()
print(cmd)


In [ ]:
# Validate that the baseline and filtered aggregate outputs exist.
required = {
    'baseline full-denominator Panel G source': BASE_INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv',
    'filtered no-ULC full-denominator Panel G source': NO_ULC_INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv',
}
for label, path in required.items():
    print(f'{label}: {path}  exists={path.exists()}')
missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required Panel G source table(s):' + chr(10) + chr(10).join(missing))


In [ ]:
# Copy source tables into the notebook work directory for easy copy-back.
all_fd = pd.read_csv(BASE_INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv', sep='	')
no_ulc_fd = pd.read_csv(NO_ULC_INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv', sep='	')
all_fd.to_csv(WORK_DIR / 'panel_g_all_models_full_denom_source.tsv', sep='	', index=False)
no_ulc_fd.to_csv(WORK_DIR / 'panel_g_no_unknown_likely_coding_full_denom_source.tsv', sep='	', index=False)
print('all_models:', all_fd.shape)
print('no_unknown_likely_coding:', no_ulc_fd.shape)
display(no_ulc_fd.head())


In [ ]:
# Quick median source check before rendering.
check = (no_ulc_fd.groupby(['direction','biotype','classification'])['n_transcripts']
         .median().reset_index(name='median_n_transcripts_per_assembly')
         .sort_values(['direction','biotype','classification']))
check.to_csv(WORK_DIR / 'panel_g_no_unknown_likely_coding_median_source_check.tsv', sep='	', index=False)
display(check.head(20))


In [ ]:
# Render Panel G all-model vs no-ULC side by side.
BIOTYPE_ORDER = ['protein_coding', 'lncRNA', 'pseudogene', 'other_ncRNA', 'other']
BIOTYPE_LABELS = {
    'protein_coding': 'Protein-coding',
    'lncRNA': 'lncRNA',
    'pseudogene': 'Pseudogene',
    'other_ncRNA': 'Other ncRNA',
    'other': 'Other coding types',
}
FULL_DENOM_GROUP_MAP = {
    'Exact_Match': 'Exact match',
    'Intron_Match': 'Same intron chain',
    'Intron_Subset': 'Partial overlap',
    'Intron_Superset': 'Partial overlap',
    'Partial_5': 'Partial overlap',
    'Partial_3': 'Partial overlap',
    'Other_Partial': 'Partial overlap',
    'No_Match': 'No match',
    'Gene_Not_Shared': 'Gene not shared',
}
GROUP_5_ORDER = ['Exact match', 'Same intron chain', 'Partial overlap', 'No match', 'Gene not shared']
GROUP_5_COLORS = {
    'Exact match': '#2ecc71',
    'Same intron chain': '#7fd37f',
    'Partial overlap': '#f1c40f',
    'No match': '#e74c3c',
    'Gene not shared': '#95a5a6',
}

def med5_counts(df, direction):
    tmp = df[df['direction'] == direction].copy()
    tmp['group'] = tmp['classification'].map(FULL_DENOM_GROUP_MAP)
    grouped = tmp.groupby(['assembly_accession','biotype','group'])['n_transcripts'].sum().reset_index()
    med = grouped.groupby(['biotype','group'])['n_transcripts'].median().reset_index()
    return med.pivot(index='biotype', columns='group', values='n_transcripts').reindex(index=BIOTYPE_ORDER, columns=GROUP_5_ORDER, fill_value=0)

def save_panel_g(fd_df, label, out_prefix):
    med_ens = med5_counts(fd_df, 'Ensembl_to_CAT')
    med_cat = med5_counts(fd_df, 'CAT_to_Ensembl')
    med = pd.concat([
        med_ens.reset_index().melt(id_vars='biotype', var_name='classification_group', value_name='median_transcript_count').assign(direction='Ensembl_to_CAT'),
        med_cat.reset_index().melt(id_vars='biotype', var_name='classification_group', value_name='median_transcript_count').assign(direction='CAT_to_Ensembl'),
    ], ignore_index=True)
    med.to_csv(WORK_DIR / f'{out_prefix}_median_counts.tsv', sep='\t', index=False)

    fig, axes = plt.subplots(1, len(BIOTYPE_ORDER), figsize=(15, 3.2), sharex=False)
    bar_height = 0.34
    y_ens = 0.5 - bar_height / 2
    y_cat = 0.5 + bar_height / 2
    for ax, biotype in zip(axes, BIOTYPE_ORDER):
        left_ens = 0
        left_cat = 0
        for grp in GROUP_5_ORDER:
            v_ens = med_ens.loc[biotype, grp]
            v_cat = med_cat.loc[biotype, grp]
            ax.barh(y_ens, v_ens, left=left_ens, height=bar_height, color=GROUP_5_COLORS[grp], edgecolor='white', linewidth=0.3)
            ax.barh(y_cat, v_cat, left=left_cat, height=bar_height, color=GROUP_5_COLORS[grp], edgecolor='white', linewidth=0.3, alpha=0.65)
            left_ens += v_ens
            left_cat += v_cat
        ax.set_title(BIOTYPE_LABELS[biotype], fontsize=9)
        ax.set_yticks([y_ens, y_cat]); ax.set_yticklabels(['Ens→CAT', 'CAT→Ens'], fontsize=7)
        ax.set_ylim(1.0, 0.0)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    fig.suptitle(label, fontsize=11, fontweight='bold', x=0.01, ha='left')
    fig.supxlabel('Median number of transcripts across assemblies', y=0.02)
    handles = [mpatches.Patch(color=GROUP_5_COLORS[g], label=g) for g in GROUP_5_ORDER]
    fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=5, fontsize=8, frameon=False)
    plt.tight_layout(rect=[0, 0.08, 1, 0.9])
    fig.savefig(WORK_DIR / 'figures' / f'{out_prefix}.png', dpi=300, bbox_inches='tight')
    fig.savefig(WORK_DIR / 'figures' / f'{out_prefix}.pdf', bbox_inches='tight')
    plt.show()

all_fd = pd.read_csv(BASE_INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv', sep='\t')
no_ulc_fd = pd.read_csv(NO_ULC_INTRON_DIR / 'intron_chain_full_denom_per_assembly.tsv', sep='\t')
all_fd.to_csv(WORK_DIR / 'panel_g_all_models_full_denom_source.tsv', sep='\t', index=False)
no_ulc_fd.to_csv(WORK_DIR / 'panel_g_no_unknown_likely_coding_full_denom_source.tsv', sep='\t', index=False)

save_panel_g(all_fd, 'Panel G: all CAT models', 'panel_g_all_models')
save_panel_g(no_ulc_fd, 'Panel G: CAT excluding unknown_likely_coding models', 'panel_g_no_unknown_likely_coding')
print('Copy back:', WORK_DIR)
